In [ ]:
!pip install nest-asyncio aiogram efficientnet_pytorch

In [ ]:
import nest_asyncio
nest_asyncio.apply()
__import__('IPython').embed()

inference 

In [ ]:
from typing import Callable, Tuple,  Any
from functools import partial
import torch
import json
import os
import albumentations as albu
import cv2
import numpy as np
import re
from torch import nn
import cv2
from PIL import Image
from torch.nn.modules.linear import Linear
from torch.nn.modules.pooling import AdaptiveAvgPool2d
from efficientnet_pytorch import EfficientNet
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from typing import Callable, Dict, Mapping, Tuple, Optional, Union
ENCODERS = {
    
    'efficientnet': {
        'features': 1408,
        'init_op': EfficientNet.from_name('efficientnet-b2', num_classes=131),
    },

}

class SignsClassifier(nn.Module):
    """
    A model for classifying signs.
    """

    def __init__(self, encoder_name: str, n_classes: int, dropout_rate: float = 0.0):
        """Initializing the class.

        :param encoder_name: name of the network encoders
        :param n_classes: number of output classes
        :param dropout_rate: dropout rate
        """
        super().__init__()
        self.encoder = ENCODERS[encoder_name]['init_op']
        self.avg_pool = AdaptiveAvgPool2d((1, 1))
        
        self.fc = Linear(ENCODERS[encoder_name]['features'], n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Getting the model prediction.

        :param x: input batch tensor
        :return: prediction
        """
        x = self.encoder(inputs=x)
        
        return x


def load_json_file(path: str) -> Any: # Loading JSON File
    with open(path, 'r') as f:
        data = json.load(f)
    return data

def prep(img, img_size: Tuple[int, int] = (224, 224)) -> Callable: # Preparing Image To Enter The Model
    valid_transform = [
        albu.Resize(img_size[0], img_size[1]),
    ]
    img = albu.Compose(valid_transform)(image=img)['image']
    img = img.astype(np.float32)
    img /= 255
    img = np.transpose(img, (2, 0, 1))
    img -= np.array([0.485, 0.456, 0.406])[:, None, None]
    img /= np.array([0.229, 0.224, 0.225])[:, None, None]

    return img

def gen_pred(imgss, class2label, MW):
    model = SignsClassifier('efficientnet', 131)
    sdict = torch.load(MW, map_location='cuda')
    model.load_state_dict(sdict['state_dict'])  # Loading 'best.pth'
    model.eval() # Setting The Module Into The Evaluation Mode
    label2class = {v : k for k, v in class2label.items()} # Converting Prediction Into Actual Class Labels
    img = Image.open(imgss) # Opening The File Using PIL
    img = np.array(img) # Converting The Image To A NumPy Array
    img = cv2.cvtColor(np.array(img), cv2.COLOR_BGR2RGB) # Converting Image From One Color Space To Another
    im = prep(img, (224, 224)) # Applying The Prep Function
    im = torch.from_numpy(im) # NumPy Array -> Tensor
    im = im.unsqueeze(0)
    pred = model(im) # Applying The Earlier Model Variable
    pred = nn.LogSoftmax(dim=1)(pred) # Using SoftMax Activation
    pred = pred.argmax(dim=-1).cpu().numpy()[0]
    pred = label2class[pred] # Converting Prediction To Class ID
    #en_pred = en_pred[pred] # Returning Prediction In English 
    #ru_pred = ru_pred[pred] # Returning Prediction In Russian 
    os.remove(imgss)
    return pred#, ru_pred, en_pred
def to_eng(pred, translate):
    newpred = []
    for i in range(len(pred.split(','))):
        if i != 0:
            newpred.append(',')
        for j in range(len(pred.split(',')[i].split('+'))):
            if j != 0:
                newpred.append('+')
            newpred.append(translate[pred.split(',')[i].split('+')[j]])
            
        
    string = ''
    for i in newpred:
        string += i
    return string

In [ ]:
import os
from aiogram import Bot, Dispatcher, executor, types
from torch import nn
from PIL import Image

In [ ]:
C2LP = 'path2class2label.json'
encodername = 'efficientnet'
MW = 'path2modelweights'

In [ ]:
class2label = load_json_file(C2LP)
translate = load_json_file('path2rus_direction_to_eng_direction.json') 

In [ ]:
os.mkdir('./photos')

In [ ]:

 # path to image
#dc_en = load_json_file(DCEN)
#dc_ru = load_json_file(DCRU)
#mkdir('/tmp'); mkdir('/tmp/logs'); mkdir('/tmp/photos')

# Объект бота
bot = Bot(token="")
# Диспетчер для бота
dp = Dispatcher(bot)

user_lang={}
@dp.message_handler(commands="start")
async def cmd_start(message: types.Message):
    keyboard = types.ReplyKeyboardMarkup(resize_keyboard=True)
    buttons = ["Русский", "English"]
    keyboard.add(*buttons)
    await message.answer("Please, choose language / Пожалуйста, выберите язык", reply_markup=keyboard)

@dp.message_handler(lambda message: message.text == "English")
async def with_puree(message: types.Message):
    user_lang[message.from_user.id] = 'en'
    await message.reply("Hello, dear customer. It is me, TwoGIS road sign classification system. This project was created for AIIJC competition. I am classifying road sign. Please, send me road sign image and wait for results. I am working with 224x224 image. But you can send me any other sized picture.")
      
@dp.message_handler(lambda message: message.text == "Русский")
async def without_puree(message: types.Message):
    user_lang[message.from_user.id] = 'ru'
    await message.reply("Здравствуйте, уважаемый. Это классификатор дорожных знаков для компании TwoGIS. Данный проект создавался в рамках состязания AIIJC. Пожалуйста, пришлите мне фото с дорожным знаком и ждите результата. Я работаю с 224х224 фотографиями, но вы можете прислать любой другой размер тоже.")
    
@dp.message_handler(content_types=["photo"])
async def download_photo(message: types.Message):
    await message.photo[-1].download(destination="./photos")
    pred = gen_pred('./photos/photos/'+os.listdir('./photos/photos')[0], class2label, MW)
    #await message.answer('я подумал о смысле жизни')
    if user_lang[message.from_user.id] is 'en':
       
        await message.answer('Successfully uploaded')
        res = ['1 line: ']
        splt = pred.split(',')
        for i in range(len(splt)):
          if i != 0:
            res.append(f'\n{i+1} line: ')
          for j in range(len(splt[i].split('+'))):
            if j != 0:
              res.append(' and ')
            res.append(to_eng(splt[i].split('+')[j], translate))
        ans = ''
        for i in res:
          ans += i
        await message.answer(ans)
    else:
        await message.answer('Успешно загружено.')
        res = ['1 полоса: ']
        splt = pred.split(',')
        for i in range(len(splt)):
          if i != 0:
            res.append(f'\n{i+1} полоса: ')
          for j in range(len(splt[i].split('+'))):
            if j != 0:
              res.append(' и ')
            res.append(splt[i].split('+')[j])
        ans = ''
        for i in res:
          ans += i
        await message.answer(ans)
   # for i in os.listdir('./photos'):
   #   os.remove('./photos/' + i) 
   # os.removedirs('./photos')

In [ ]:
if __name__ == "__main__":
    # Запуск бота
    executor.start_polling(dp, skip_updates=True)